# Corte 3, Punto 4: tablas de consumo del dashboard

Por: Gabriel Rodriguez y Jonatan Palomares

**Subtareas:** T4.1 (tablas de consumo) y T4.2 (tecnología y forma de ejecución).

Entrada: gold (con las marcas de calidad de T1.5), los nombres de las tablas resumen de T1.4 y las ventanas del flujo (notebook 06). Salidas: `procesado/dashboard/` en MinIO y los mismos datos en la carpeta `dashboard/data/` del repositorio, que es lo que lee el dashboard.

Este notebook es el único paso del dashboard que usa Spark. El dashboard solo lee los archivos de `dashboard/data/`.

## Configuración

Las rutas están en `src/pamd/config.py` y la sesión de Spark con MinIO en `src/pamd/spark_session.py`. Las credenciales vienen de las variables de entorno definidas en `.env`.

In [1]:
# Importación de librerías

import json
from datetime import datetime, timezone
from pathlib import Path
from time import perf_counter
from IPython.display import display, Markdown

import pandas as pd

from pyspark.sql import functions as F
from pyspark import StorageLevel

from pamd.config import (
    GOLD_PARQUET,
    SILVER_PARQUET,
    PROCESADO_ENTIDAD_PARQUET,
    PROCESADO_CONTRATISTA_PARQUET,
    STREAMING_VENTANAS,
    STREAMING_TARDIOS,
    DASHBOARD_SERIE_PARQUET,
    DASHBOARD_CONTRATISTAS_MES_PARQUET,
    DASHBOARD_ENTIDADES_PARQUET,
    DASHBOARD_CONTRATISTAS_PARQUET,
    DASHBOARD_VENTANAS_PARQUET,
    WRITE_MODE,
)
from pamd.spark_session import get_spark
from pamd.calidad import marcar_atipicos, unificar_nit_entidad
from pamd.flujo import ESQUEMA_FLUJO, CLAVES_VENTANA, agregar_por_lotes, existe

# Spark conectado a MinIO; las credenciales se leen del archivo .env
spark = get_spark()
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")  # toPandas() más rápido al exportar

# Carpeta del repositorio que lee el dashboard (el notebook se ejecuta desde notebooks/)
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CARPETA_LOCAL = RAIZ / "dashboard" / "data"

## T4.2 Tecnología y forma de ejecución

**Decisión: Streamlit.** El proyecto ya está en Python, los filtros y gráficos se escriben en un solo archivo y el dashboard se abre con un comando (`streamlit run dashboard/app.py`). Lee los archivos de `dashboard/data/` con pandas, así que no necesita Spark ni MinIO en marcha. Las instrucciones de arranque están en `dashboard/README.md`.

Se descartó una página web con JavaScript porque filtrar medio millón de filas en el navegador exige más código para el mismo resultado, y Dash porque pide más estructura (callbacks) que Streamlit para tres módulos.

## T4.1 Tablas de consumo

Las tablas resumen de T1.4 agrupan por **una** dimensión (entidad, contratista, departamento o mes). El dashboard filtra por departamento, entidad y fecha **a la vez**, así que necesita tablas cruzadas. Se calculan desde gold con las mismas reglas de T1.4 (`src/pamd/calidad.py`) y las mismas medidas, con los atípicos separados del valor normal.

| Tabla | Una fila por | Módulos que la usan |
|---|---|---|
| `serie_mensual` | mes × departamento × entidad | tarjetas de totales, gráfico temporal, ranking de entidades, valor por departamento y concentración |
| `contratistas_mensual` | mes × departamento × entidad × contratista | ranking de contratistas con los filtros aplicados |
| `entidades` | entidad | nombre que se muestra en el filtro y en el ranking |
| `contratistas` | contratista | nombre que se muestra en el ranking |
| `ventanas_flujo` | mes × departamento | comprobar que el gráfico temporal coincide con las ventanas del flujo (T2.4) |
| `metadatos.json` | (un solo objeto) | totales de gold para comprobar y fecha del último lote procesado |

El ranking de entidades, el ranking de contratistas y el valor por departamento que pide el plan no se guardan como tablas aparte: son agrupaciones de `serie_mensual` y `contratistas_mensual` que el dashboard calcula después de aplicar los filtros. Guardarlas ya agrupadas daría rankings que no responden a los filtros.

`serie_mensual` se podría obtener agrupando `contratistas_mensual`, pero se guarda aparte porque es unas cinco veces más pequeña y la usan casi todos los módulos.

**Valores en pesos enteros.** Los valores de gold no tienen centavos, así que se guardan como enteros de 64 bits en lugar de `decimal`. pandas suma enteros de forma exacta y rápida; con `decimal` cada valor sería un objeto de Python. La celda siguiente comprueba que no hay centavos y que el total cabe en 64 bits.

In [2]:
# T4.1 · Gold con las marcas de T1.5 y el departamento como en T1.4
SIN_DEPARTAMENTO = "SIN DEPARTAMENTO"

gold = spark.read.parquet(GOLD_PARQUET)
silver = spark.read.parquet(SILVER_PARQUET)
valor = F.col("valor_total_adjudicacion")

gold_marcado = (
    marcar_atipicos(unificar_nit_entidad(gold), silver)
    .withColumn(
        "departamento",
        F.when(F.trim("departamento_entidad") != "", F.trim("departamento_entidad")).otherwise(F.lit(SIN_DEPARTAMENTO)),
    )
    .select("anio_mes", "departamento", "id_entidad", "id_contratista", "fecha_adjudicacion",
            "valor_total_adjudicacion", "es_atipico", "es_valor_cero")
    .persist(StorageLevel.MEMORY_AND_DISK)
)

tot_gold = gold_marcado.agg(
    F.count("*").alias("contratos"),
    F.sum(F.col("es_atipico").cast("int")).alias("atipicos"),
    F.sum(F.col("es_valor_cero").cast("int")).alias("ceros"),
    F.sum(valor).alias("valor"),
    F.sum(F.when(~F.col("es_atipico"), valor).otherwise(0)).alias("valor_total"),
    F.sum(F.when(F.col("es_atipico"), valor).otherwise(0)).alias("valor_atipicos"),
    F.sum((valor != F.floor(valor)).cast("int")).alias("con_centavos"),
    F.min("anio_mes").alias("primer_mes"),
    F.max("anio_mes").alias("ultimo_mes"),
    F.max("fecha_adjudicacion").alias("ultima_fecha"),
).first()

assert tot_gold["con_centavos"] == 0, "Hay valores con centavos: no se pueden guardar como pesos enteros"
assert tot_gold["valor"] < 2**63, "El valor total no cabe en un entero de 64 bits"
print(f"Gold: {tot_gold['contratos']:,} contratos de {tot_gold['primer_mes']} a {tot_gold['ultimo_mes']}, "
      f"{tot_gold['atipicos']:,} atípicos, {int(tot_gold['valor']):,} pesos.")

Gold: 530,192 contratos de 2015-06 a 2026-09, 432 atípicos, 175,067,685,240,657,754 pesos.


In [3]:
# T4.1 · Tablas cruzadas: las medidas de T1.4, con los valores como pesos enteros
MEDIDAS = ["contratos", "contratos_atipicos", "contratos_valor_cero", "valor_total", "valor_atipicos"]

def resumir_consumo(df, claves):
    """groupBy().agg() sobre gold marcado, con las medidas de T1.4 (los atípicos se cuentan y se suman aparte)."""
    normal = ~F.col("es_atipico")
    return df.groupBy(*claves).agg(
        F.count("*").alias("contratos"),
        F.sum(F.col("es_atipico").cast("long")).alias("contratos_atipicos"),
        F.sum(F.col("es_valor_cero").cast("long")).alias("contratos_valor_cero"),
        F.sum(F.when(normal, valor).otherwise(0)).cast("long").alias("valor_total"),
        F.sum(F.when(~normal, valor).otherwise(0)).cast("long").alias("valor_atipicos"),
    )

CLAVES = {
    "serie_mensual": ["anio_mes", "departamento", "id_entidad"],
    "contratistas_mensual": ["anio_mes", "departamento", "id_entidad", "id_contratista"],
    "entidades": ["id_entidad"],
    "contratistas": ["id_contratista"],
    "ventanas_flujo": ["anio_mes", "departamento"],
}

serie_df = resumir_consumo(gold_marcado, CLAVES["serie_mensual"])
contratistas_mes_df = resumir_consumo(gold_marcado, CLAVES["contratistas_mensual"])

# Nombres: los de las tablas resumen de T1.4 (el más frecuente de cada clave), para que coincidan con ellas
entidades_df = spark.read.parquet(PROCESADO_ENTIDAD_PARQUET).select(
    "id_entidad", F.coalesce("nombre_entidad", "id_entidad").alias("nombre_entidad"), "departamento_entidad")
contratistas_df = spark.read.parquet(PROCESADO_CONTRATISTA_PARQUET).select(
    "id_contratista", F.coalesce("nombre_contratista", "id_contratista").alias("nombre_contratista"))

### Ventanas del flujo

El resultado del flujo es el mismo de T2.4: la tabla de ventanas que deja el destino idempotente del notebook 06 (`streaming/ventanas/tabla`) más los contratos que llegaron fuera de la tolerancia y se registraron aparte (`streaming/tardios`, T2.5). Se exporta para que el gráfico temporal se pueda comparar con él (criterio 3 de T4.4).

De `streaming/ventanas/cambios` sale el último micro-lote que procesó el flujo y la fecha en que se escribió; el dashboard la muestra como fecha del último lote procesado (T4.7).

In [4]:
# T4.1 · Ventanas del flujo (tabla del destino idempotente + tardíos registrados) y último micro-lote
tabla_flujo = spark.read.parquet(f"{STREAMING_VENTANAS}/tabla").select(*CLAVES_VENTANA, *MEDIDAS)
tardios = (spark.read.schema(ESQUEMA_FLUJO).parquet(STREAMING_TARDIOS) if existe(spark, STREAMING_TARDIOS)
           else spark.createDataFrame([], ESQUEMA_FLUJO))
n_tardios = tardios.count()

ventanas_df = (
    tabla_flujo.unionByName(agregar_por_lotes(tardios, SIN_DEPARTAMENTO))
    .groupBy(*CLAVES_VENTANA).agg(*[F.sum(m).cast("long").alias(m) for m in MEDIDAS])
    .select(F.date_format("mes", "yyyy-MM").alias("anio_mes"), "departamento", *MEDIDAS)
)

# Último micro-lote: la carpeta batch_id más alta de cambios/ y la fecha de escritura de sus archivos
jvm = spark.sparkContext._jvm
ruta_cambios = jvm.org.apache.hadoop.fs.Path(f"{STREAMING_VENTANAS}/cambios")
fs = ruta_cambios.getFileSystem(spark.sparkContext._jsc.hadoopConfiguration())
lotes = {int(s.getPath().getName().split("=")[1]): s.getPath()
         for s in fs.listStatus(ruta_cambios) if s.getPath().getName().startswith("batch_id=")}
ultimo_lote = max(lotes)
escrito_ms = max(s.getModificationTime() for s in fs.listStatus(lotes[ultimo_lote]))
fecha_ultimo_lote = datetime.fromtimestamp(escrito_ms / 1000, tz=timezone.utc)

print(f"Ventanas del flujo: {tabla_flujo.count():,} en la tabla, más {n_tardios:,} contratos tardíos registrados aparte.")
print(f"Último micro-lote procesado: {ultimo_lote} ({len(lotes)} micro-lotes con cambios), escrito el {fecha_ultimo_lote:%Y-%m-%d %H:%M} UTC.")

Ventanas del flujo: 3,735 en la tabla, más 866 contratos tardíos registrados aparte.
Último micro-lote procesado: 40 (21 micro-lotes con cambios), escrito el 2026-11-16 23:17 UTC.


### Escritura en MinIO y exportación a `dashboard/data/`

`RECONSTRUIR_CONSUMO = True` recalcula las tablas, las escribe en `procesado/dashboard/` (un archivo Parquet por tabla) y las exporta a `dashboard/data/`; con `False` se reutiliza lo que ya está en MinIO y solo se comprueba lo exportado.

La exportación parte de lo que se leyó de MinIO, no del cálculo en memoria: así el archivo local es una copia de lo que quedó guardado. Las filas van ordenadas por su clave, para que dos ejecuciones produzcan el mismo archivo.

In [5]:
# T4.1 · Escritura de las tablas de consumo en MinIO y exportación local (un archivo por tabla)
# True la primera vez (o cuando cambie gold o el flujo); False reutiliza lo que ya está en MinIO y en dashboard/data/.
RECONSTRUIR_CONSUMO = False

RUTAS = {
    "serie_mensual": DASHBOARD_SERIE_PARQUET,
    "contratistas_mensual": DASHBOARD_CONTRATISTAS_MES_PARQUET,
    "entidades": DASHBOARD_ENTIDADES_PARQUET,
    "contratistas": DASHBOARD_CONTRATISTAS_PARQUET,
    "ventanas_flujo": DASHBOARD_VENTANAS_PARQUET,
}

if RECONSTRUIR_CONSUMO:
    por_escribir = {
        "serie_mensual": serie_df,
        "contratistas_mensual": contratistas_mes_df,
        "entidades": entidades_df,
        "contratistas": contratistas_df,
        "ventanas_flujo": ventanas_df,
    }
    for nombre, df in por_escribir.items():
        df.coalesce(1).write.mode(WRITE_MODE).option("compression", "snappy").parquet(RUTAS[nombre])

consumo = {nombre: spark.read.parquet(ruta) for nombre, ruta in RUTAS.items()}

if RECONSTRUIR_CONSUMO:
    CARPETA_LOCAL.mkdir(parents=True, exist_ok=True)
    for nombre, df in consumo.items():
        local = df.toPandas().sort_values(CLAVES[nombre]).reset_index(drop=True)
        local.to_parquet(CARPETA_LOCAL / f"{nombre}.parquet", index=False, compression="zstd")

    metadatos = {
        "generado_utc": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M"),
        "fuente": "gold (s3a://rawdata/gold/secop/adjudicaciones) con las reglas de src/pamd/calidad.py",
        "gold": {
            "contratos": tot_gold["contratos"],
            "contratos_atipicos": tot_gold["atipicos"],
            "contratos_valor_cero": tot_gold["ceros"],
            "valor_total": int(tot_gold["valor_total"]),
            "valor_atipicos": int(tot_gold["valor_atipicos"]),
            "primer_mes": tot_gold["primer_mes"],
            "ultimo_mes": tot_gold["ultimo_mes"],
            "ultima_fecha_adjudicacion": str(tot_gold["ultima_fecha"]),
        },
        "flujo": {
            "ultimo_micro_lote": ultimo_lote,
            "micro_lotes_con_cambios": len(lotes),
            "fecha_ultimo_micro_lote_utc": fecha_ultimo_lote.strftime("%Y-%m-%d %H:%M"),
            "contratos_tardios_registrados": n_tardios,
        },
        "filas": {nombre: df.count() for nombre, df in consumo.items()},
    }
    (CARPETA_LOCAL / "metadatos.json").write_text(json.dumps(metadatos, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

print("Archivos en", CARPETA_LOCAL.relative_to(RAIZ).as_posix() + ":", ", ".join(sorted(p.name for p in CARPETA_LOCAL.iterdir())))

Archivos en dashboard/data: contratistas.parquet, contratistas_mensual.parquet, entidades.parquet, metadatos.json, serie_mensual.parquet, ventanas_flujo.parquet


### Esquema de las tablas de consumo

Desde aquí no se usa Spark: los archivos se leen con pandas, igual que hará el dashboard.

In [6]:
# T4.1 · Esquema documentado de cada tabla, leída con pandas desde dashboard/data/
significado = {
    "anio_mes": "Año y mes de adjudicación, 'AAAA-MM'.",
    "departamento": "Departamento de la entidad contratante en cada contrato; sin departamento va a 'SIN DEPARTAMENTO'.",
    "id_entidad": "Identificador de la entidad contratante (misma regla que gold, con los NIT de dígito de verificación pegado unificados).",
    "id_contratista": "Identificador del contratista (NIT, luego código, luego nombre).",
    "nombre_entidad": "Nombre más frecuente entre los contratos de la entidad (tabla resumen de T1.4).",
    "departamento_entidad": "Departamento más frecuente entre los contratos de la entidad; solo descriptivo, los filtros usan 'departamento'.",
    "nombre_contratista": "Nombre más frecuente entre los contratos del contratista (tabla resumen de T1.4).",
    "contratos": "Número de contratos adjudicados, con los atípicos incluidos.",
    "contratos_atipicos": "Contratos marcados como atípicos en T1.5 (valor inconsistente con el precio base o extremo).",
    "contratos_valor_cero": "Contratos con valor 0.",
    "valor_total": "Suma del valor adjudicado sin los atípicos, en pesos enteros.",
    "valor_atipicos": "Suma del valor adjudicado de los contratos atípicos, en pesos enteros.",
}

tablas, segundos = {}, {}
for nombre in RUTAS:
    inicio = perf_counter()
    tablas[nombre] = pd.read_parquet(CARPETA_LOCAL / f"{nombre}.parquet")
    segundos[nombre] = perf_counter() - inicio
metadatos = json.loads((CARPETA_LOCAL / "metadatos.json").read_text(encoding="utf-8"))

for nombre, t in tablas.items():
    esquema = pd.DataFrame(
        [(c, str(t[c].dtype), "clave" if c in CLAVES[nombre] else "", significado.get(c, "")) for c in t.columns],
        columns=["columna", "tipo", "clave", "significado"],
    )
    assert (esquema["significado"] != "").all(), f"Hay columnas sin documentar en {nombre}"
    display(Markdown(f"**`{nombre}.parquet`** (en MinIO: `{RUTAS[nombre]}`)"))
    display(esquema.style.hide(axis="index"))

display(Markdown("**`metadatos.json`**"))
print(json.dumps(metadatos, ensure_ascii=False, indent=2))

**`serie_mensual.parquet`** (en MinIO: `s3a://rawdata/procesado/dashboard/serie_mensual`)

columna,tipo,clave,significado
anio_mes,object,clave,"Año y mes de adjudicación, 'AAAA-MM'."
departamento,object,clave,Departamento de la entidad contratante en cada contrato; sin departamento va a 'SIN DEPARTAMENTO'.
id_entidad,object,clave,"Identificador de la entidad contratante (misma regla que gold, con los NIT de dígito de verificación pegado unificados)."
contratos,int64,,"Número de contratos adjudicados, con los atípicos incluidos."
contratos_atipicos,int64,,Contratos marcados como atípicos en T1.5 (valor inconsistente con el precio base o extremo).
contratos_valor_cero,int64,,Contratos con valor 0.
valor_total,int64,,"Suma del valor adjudicado sin los atípicos, en pesos enteros."
valor_atipicos,int64,,"Suma del valor adjudicado de los contratos atípicos, en pesos enteros."


**`contratistas_mensual.parquet`** (en MinIO: `s3a://rawdata/procesado/dashboard/contratistas_mensual`)

columna,tipo,clave,significado
anio_mes,object,clave,"Año y mes de adjudicación, 'AAAA-MM'."
departamento,object,clave,Departamento de la entidad contratante en cada contrato; sin departamento va a 'SIN DEPARTAMENTO'.
id_entidad,object,clave,"Identificador de la entidad contratante (misma regla que gold, con los NIT de dígito de verificación pegado unificados)."
id_contratista,object,clave,"Identificador del contratista (NIT, luego código, luego nombre)."
contratos,int64,,"Número de contratos adjudicados, con los atípicos incluidos."
contratos_atipicos,int64,,Contratos marcados como atípicos en T1.5 (valor inconsistente con el precio base o extremo).
contratos_valor_cero,int64,,Contratos con valor 0.
valor_total,int64,,"Suma del valor adjudicado sin los atípicos, en pesos enteros."
valor_atipicos,int64,,"Suma del valor adjudicado de los contratos atípicos, en pesos enteros."


**`entidades.parquet`** (en MinIO: `s3a://rawdata/procesado/dashboard/entidades`)

columna,tipo,clave,significado
id_entidad,object,clave,"Identificador de la entidad contratante (misma regla que gold, con los NIT de dígito de verificación pegado unificados)."
nombre_entidad,object,,Nombre más frecuente entre los contratos de la entidad (tabla resumen de T1.4).
departamento_entidad,object,,"Departamento más frecuente entre los contratos de la entidad; solo descriptivo, los filtros usan 'departamento'."


... (2 outputs left out)

**`ventanas_flujo.parquet`** (en MinIO: `s3a://rawdata/procesado/dashboard/ventanas_flujo`)

columna,tipo,clave,significado
anio_mes,object,clave,"Año y mes de adjudicación, 'AAAA-MM'."
departamento,object,clave,Departamento de la entidad contratante en cada contrato; sin departamento va a 'SIN DEPARTAMENTO'.
contratos,int64,,"Número de contratos adjudicados, con los atípicos incluidos."
contratos_atipicos,int64,,Contratos marcados como atípicos en T1.5 (valor inconsistente con el precio base o extremo).
contratos_valor_cero,int64,,Contratos con valor 0.
valor_total,int64,,"Suma del valor adjudicado sin los atípicos, en pesos enteros."
valor_atipicos,int64,,"Suma del valor adjudicado de los contratos atípicos, en pesos enteros."


**`metadatos.json`**

{
  "generado_utc": "2026-11-17 01:07",
  "fuente": "gold (s3a://rawdata/gold/secop/adjudicaciones) con las reglas de src/pamd/calidad.py",
  "gold": {
    "contratos": 530192,
    "contratos_atipicos": 432,
    "contratos_valor_cero": 1527,
    "valor_total": 352909018925397,
    "valor_atipicos": 174714776221732357,
    "primer_mes": "2015-06",
    "ultimo_mes": "2026-09",
    "ultima_fecha_adjudicacion": "2026-09-01"
  },
  "flujo": {
    "ultimo_micro_lote": 40,
    "micro_lotes_con_cambios": 21,
    "fecha_ultimo_micro_lote_utc": "2026-11-16 23:17",
    "contratos_tardios_registrados": 866
  },
  "filas": {
    "serie_mensual": 99789,
    "contratistas_mensual": 481397,
    "entidades": 4050,
    "contratistas": 99606,
    "ventanas_flujo": 3735
  }
}


### Criterios de aceptación

1. Las tablas se cargan sin Spark y en pocos segundos (se mide la lectura con pandas de los cinco archivos).
2. Cada tabla tiene el esquema documentado (celda anterior).
3. Sus totales coinciden con gold: contratos, contratos atípicos, contratos con valor cero, `valor_total` y `valor_atipicos`, en pesos exactos. Además, cada tabla tiene una sola fila por clave, sin claves nulas; todas las entidades y contratistas tienen nombre; y `serie_mensual` agrupada por mes y departamento es igual, ventana por ventana, a las ventanas del flujo.

In [7]:
# T4.1 · Criterios de aceptación, comprobados con pandas sobre los archivos exportados
g = metadatos["gold"]
esperado = pd.Series({"contratos": g["contratos"], "contratos_atipicos": g["contratos_atipicos"],
                      "contratos_valor_cero": g["contratos_valor_cero"], "valor_total": g["valor_total"],
                      "valor_atipicos": g["valor_atipicos"]})
# Los totales de metadatos.json son los de gold calculados en esta ejecución (si se reconstruyó) o en la anterior
assert g["contratos"] == tot_gold["contratos"] and g["valor_total"] + g["valor_atipicos"] == int(tot_gold["valor"]), \
    "metadatos.json no corresponde al gold actual: ejecuta con RECONSTRUIR_CONSUMO = True"

lista = []
for nombre, t in tablas.items():
    archivo = CARPETA_LOCAL / f"{nombre}.parquet"
    assert not t[CLAVES[nombre]].isna().any().any(), f"{nombre}: hay claves nulas"
    assert not t.duplicated(CLAVES[nombre]).any(), f"{nombre}: hay claves repetidas"
    assert len(t) == metadatos["filas"][nombre], f"{nombre}: el archivo local no tiene las filas de MinIO"
    if "contratos" in t.columns:
        assert (t[MEDIDAS].dtypes == "int64").all(), f"{nombre}: las medidas deben ser enteros de 64 bits"
        assert t[MEDIDAS].sum().equals(esperado), f"{nombre}: los totales no coinciden con gold"
    lista.append({"tabla": nombre, "archivo": archivo.relative_to(RAIZ).as_posix(), "filas": len(t), "columnas": t.shape[1],
                  "tamaño (KB)": archivo.stat().st_size / 1024, "carga con pandas (s)": segundos[nombre]})

# Nombres para todas las claves que aparecen en las tablas cruzadas
assert set(tablas["serie_mensual"]["id_entidad"]) == set(tablas["entidades"]["id_entidad"]), "Entidades sin nombre o sin contratos"
assert set(tablas["contratistas_mensual"]["id_contratista"]) == set(tablas["contratistas"]["id_contratista"]), "Contratistas sin nombre o sin contratos"
assert not tablas["entidades"]["nombre_entidad"].isna().any() and not tablas["contratistas"]["nombre_contratista"].isna().any()

# serie_mensual por mes y departamento = ventanas del flujo, ventana por ventana
claves_ventana = CLAVES["ventanas_flujo"]
desde_serie = tablas["serie_mensual"].groupby(claves_ventana)[MEDIDAS].sum()
desde_flujo = tablas["ventanas_flujo"].set_index(claves_ventana)[MEDIDAS].sort_index()
assert desde_serie.equals(desde_flujo), "serie_mensual no coincide con las ventanas del flujo"

# contratistas_mensual agrupada sin el contratista = serie_mensual
claves_serie = CLAVES["serie_mensual"]
assert tablas["contratistas_mensual"].groupby(claves_serie)[MEDIDAS].sum().equals(
    tablas["serie_mensual"].set_index(claves_serie)[MEDIDAS].sort_index()), "contratistas_mensual no suma serie_mensual"

carga_total = sum(segundos.values())
assert carga_total < 5, f"La carga tarda {carga_total:.1f} s"

display(pd.DataFrame(lista).style.hide(axis="index").format(
    {"filas": "{:,}", "tamaño (KB)": "{:,.0f}", "carga con pandas (s)": "{:.2f}"}))
tamano_total = sum(p.stat().st_size for p in CARPETA_LOCAL.iterdir()) / 1024**2
print(f"Total: {tamano_total:.1f} MB en disco; los cinco archivos se cargan con pandas en {carga_total:.2f} s, sin Spark.")
print(f"Totales de cada tabla = gold: {g['contratos']:,} contratos, {g['contratos_atipicos']:,} atípicos, "
      f"{g['valor_total']:,} pesos sin atípicos y {g['valor_atipicos']:,} pesos de atípicos.")
print(f"Ventanas (mes x departamento) iguales a las del flujo: {len(desde_flujo):,}.")
print("Criterios de aceptación de T4.1: OK")

tabla,archivo,filas,columnas,tamaño (KB),carga con pandas (s)
serie_mensual,dashboard/data/serie_mensual.parquet,"99,789",8,776,0.13
contratistas_mensual,dashboard/data/contratistas_mensual.parquet,"481,397",9,"4,151",0.38
entidades,dashboard/data/entidades.parquet,"4,050",3,81,0.02
contratistas,dashboard/data/contratistas.parquet,"99,606",2,"1,844",0.14
ventanas_flujo,dashboard/data/ventanas_flujo.parquet,"3,735",7,42,0.02


Total: 6.7 MB en disco; los cinco archivos se cargan con pandas en 0.69 s, sin Spark.
Totales de cada tabla = gold: 530,192 contratos, 432 atípicos, 352,909,018,925,397 pesos sin atípicos y 174,714,776,221,732,357 pesos de atípicos.
Ventanas (mes x departamento) iguales a las del flujo: 3,735.
Criterios de aceptación de T4.1: OK


### Vista rápida

Lo que hará el dashboard con estas tablas: aplicar un filtro y agrupar. Aquí, sin filtros, las cinco entidades y los cinco contratistas con más valor adjudicado (sin atípicos).

In [8]:
# T4.1 · Vista rápida con pandas: top 5 de entidades y de contratistas, sin filtros
top_entidades = (
    tablas["serie_mensual"].groupby("id_entidad")[["contratos", "valor_total"]].sum()
    .nlargest(5, "valor_total").join(tablas["entidades"].set_index("id_entidad")["nombre_entidad"])
)
top_contratistas = (
    tablas["contratistas_mensual"].groupby("id_contratista")[["contratos", "valor_total"]].sum()
    .nlargest(5, "valor_total").join(tablas["contratistas"].set_index("id_contratista")["nombre_contratista"])
)
formato = {"contratos": "{:,}", "valor_total": "{:,}"}
display(top_entidades[["nombre_entidad", "contratos", "valor_total"]].style.format(formato))
display(top_contratistas[["nombre_contratista", "contratos", "valor_total"]].style.format(formato))

,nombre_entidad,contratos,valor_total
id_entidad,,,
NIT:800215807,INVIAS,"5,285","22,256,198,017,874"
NIT:899999061,(Secretaría Distrital de Integración Social),"18,126","19,436,497,397,090"
NIT:890905211,DISTRITO ESPECIAL DE CIENCIA TECNOLOGIA E INNOVACION DE MEDELLIN,"4,879","15,980,884,831,288"
NIT:899999081,INSTITUTO DE DESARROLLO URBANO,864,"11,453,591,009,173"
NIT:900475780,UNP,615,"8,226,967,368,610"


,nombre_contratista,contratos,valor_total
id_contratista,,,
NIT:900082143,GECELCA S.A. E.S.P.,1,"4,205,027,751,839"
NIT:8060088733,"Corporacion de Ciencia y Tecnologia para el Desarrollo de la Industria Naval, Maritima y Fluvial",112,"3,001,973,356,929"
NIT:830040256,HOSMIL,19,"2,613,029,607,356"
NIT:800223337,EMPRESA DE DESARROLLO URBANO DE MEDELLIN,96,"2,208,287,836,785"
NIT:901965187,UNION TEMPORAL INTEGRACION LOGISTICA ELECTORAL 2026,1,"2,124,806,917,129"


### Qué sigue

Con estas tablas y la decisión de T4.2, el dashboard (`dashboard/app.py`) se construye en T4.3 a T4.6: filtros y tarjetas, gráfico temporal, rankings y módulo geográfico. Si gold o el flujo cambian, hay que volver a ejecutar este notebook con `RECONSTRUIR_CONSUMO = True` para actualizar `dashboard/data/`.